In [4]:
#!/usr/bin/env python3

In [5]:
"""
TFT Set 18 (Enchanted Wilds) Data Miner -- Units, Traits, Items & Augments
============================================================================
 
Pulls the current Teamfight Tactics dataset from Riot's community-run
data mirror and extracts everything playable from a given TFT set --
by default, Set 18 "Enchanted Wilds":
 
  * Units (champions): cost, traits, base stats, ability text/scaling
  * Traits: description, breakpoint tiers, per-tier bonuses
  * Items (components + completed items + artifacts/emblems): recipe,
    description with numbers filled in, associated/incompatible traits
  * Augments: description with numbers filled in, associated traits

 Data source: https://raw.communitydragon.org/latest/cdragon/tft/en_us.json

 A note on how items/augments are isolated to one set
------------------------------------------------------
Unlike champions and traits, which cdragon conveniently pre-splits
per set (under the "sets" -> "<number>" key), items and augments all
live together in one big top-level "items" list covering *every* TFT
set that has ever existed. To isolate Set 18's items and augments,
this script:
 
  1. Looks at the leading "TFT<number>_" in each entry's apiName.
     Augments are always tied to the set that introduced them, so an
     apiName prefix of "TFT18_" means it's a Set 18 augment.
  2. Equipment items (components/completed items/artifacts) that have
     NO numbered prefix at all (e.g. "TFT_Item_InfinityEdge") are the
     "universal" item pool that's available across every set, so
     those are always included regardless of which set you ask for.
  3. Equipment items that DO carry a set-numbered prefix (radiant
     items, support items, set-specific artifacts/emblems) are
     included only when the number matches the requested set.
 
This mirrors how community trackers do it and gets the right answer
in the overwhelming majority of cases, but it's a heuristic rather
than something cdragon labels explicitly -- if a patch changes the
naming convention, adjust `item_set_number()` accordingly.
 
Usage
-----
    python tft_set18_scraper.py                    # Set 18, ./tft_data
    python tft_set18_scraper.py --set 17            # a different set
    python tft_set18_scraper.py --refresh           # force re-download
    python tft_set18_scraper.py --output-dir out    # custom output dir
 
Outputs (written to the output directory):
    cdragon_tft_raw.json    - the raw upstream payload (cached locally;
                               delete it or pass --refresh after a patch)
    set{N}_units.json       - full structured detail for every unit
    set{N}_units.csv        - flattened, spreadsheet-friendly summary
    set{N}_traits.json      - full structured detail for every trait
    set{N}_items.json       - full structured detail for every item
    set{N}_items.csv        - flattened, spreadsheet-friendly summary
    set{N}_augments.json    - full structured detail for every augment
    set{N}_augments.csv     - flattened, spreadsheet-friendly summary
 
No third-party packages required (standard library only).
"""

'\nTFT Set 18 (Enchanted Wilds) Data Miner -- Units, Traits, Items & Augments\n============================================================================\n \nPulls the current Teamfight Tactics dataset from Riot\'s community-run\ndata mirror and extracts everything playable from a given TFT set --\nby default, Set 18 "Enchanted Wilds":\n \n  * Units (champions): cost, traits, base stats, ability text/scaling\n  * Traits: description, breakpoint tiers, per-tier bonuses\n  * Items (components + completed items + artifacts/emblems): recipe,\n    description with numbers filled in, associated/incompatible traits\n  * Augments: description with numbers filled in, associated traits\n\n Data source: https://raw.communitydragon.org/latest/cdragon/tft/en_us.json\n\n A note on how items/augments are isolated to one set\n------------------------------------------------------\nUnlike champions and traits, which cdragon conveniently pre-splits\nper set (under the "sets" -> "<number>" key), items 